# Introducción a LLMs (desde cero)

> **Objetivo de este notebook:** Entender qué es un LLM, cómo conectarte a uno mediante una API y hacer tus primeras llamadas. Al terminar, habrás enviado tu primer prompt a un modelo de IA y recibido una respuesta.

---

## 1. ¿Qué es un LLM?

Un **LLM (Large Language Model)** es un modelo estadístico entrenado con enormes cantidades de texto que aprende a **predecir el siguiente token** (fragmento de palabra) dado un contexto.

### Cómo funciona en una línea:
```
Entrada (texto) → [LLM: predice token a token] → Salida (más texto)
```

### Lo que NO hace un LLM:
- ❌ No **entiende** como un humano — opera sobre patrones estadísticos
- ❌ No **busca en internet** (a menos que tenga herramientas externas)
- ❌ No tiene **memoria** entre conversaciones distintas
- ❌ No **siempre dice la verdad** — puede alucinar (inventar información)

### Lo que SÍ hace:
- ✅ Genera texto coherente y contextualizado
- ✅ Sigue instrucciones complejas en lenguaje natural
- ✅ Traduce, resume, clasifica, escribe código
- ✅ Adapta su estilo y tono según el prompt

### Ejemplos de LLMs conocidos:
| Modelo | Empresa | Acceso |
|--------|---------|--------|
| GPT-4 | OpenAI | API de pago |
| Claude | Anthropic | API de pago |
| Gemini / Gemma | Google | API (gratuita con límites) |
| LLaMA | Meta | Open source (descargable) |
| Mistral | Mistral AI | Open source |

> **En este notebook usamos Gemma (de Google) a través de la API de Gemini.** Es gratuito con límites y no requiere tarjeta.

## 2. Cargar la API Key

Para usar un LLM vía API necesitas una **clave de autenticación** (API Key). Es como una contraseña que identifica quién hace la petición.

### Buenas prácticas de seguridad:
- 🔒 **Nunca** escribas la clave directamente en el código — alguien podría verla si compartes el notebook
- ✅ Guárdala en un archivo `.env` (que NO se sube a GitHub)
- ✅ Cárgala con `python-dotenv`

### Estructura del archivo `.env`:
```
Gemini_API_Key_001=tu_clave_aqui
```

### Cómo obtener tu clave de Gemini:
1. Ve a [https://aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)
2. Haz clic en **Create API Key**
3. Cópiala y pégala en tu `.env`

In [5]:
# Importamos las librerías necesarias
from dotenv import load_dotenv  # Para leer el archivo .env
import os                       # Para acceder a las variables de entorno

# load_dotenv() busca el archivo .env y carga sus variables en el entorno del sistema
# dotenv_path: ruta al archivo .env (ajústala a tu ruta)
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# os.getenv() recupera el valor de la variable de entorno
# Si no existe, devuelve None (no da error)
api_key = os.getenv("Gemini_API_Key_001")

# Verificamos que se cargó (sin mostrar la clave real)
if api_key:
    print("✅ API Key cargada correctamente")
else:
    print("❌ API Key no encontrada. Revisa tu archivo .env")

✅ API Key cargada correctamente


> **Nota:** Este paso solo carga la clave en memoria. Todavía no nos hemos conectado al modelo.

## 3. Instalar la librería cliente

Para hablar con la API de Gemini necesitamos la librería oficial de Google: `google-genai`.

- `!pip install` ejecuta un comando del sistema desde dentro del notebook
- `-U` significa "actualizar si ya está instalada"
- Solo necesitas ejecutar esta celda una vez por entorno

In [13]:
# Instala la librería oficial de Google para usar sus modelos de IA
# El signo ! indica que es un comando de shell (terminal), no Python
!pip install -U google-genai

## 4. Conectarse al modelo y hacer la primera llamada

Ahora sí nos conectamos al LLM y le enviamos nuestra primera pregunta.

### Flujo completo:
```
Tu código → API de Gemini (internet) → Modelo (en servidores de Google) → Respuesta → Tu código
```

### Conceptos clave:
- **`genai.Client`**: el objeto que gestiona la conexión con la API
- **`client.models.generate_content()`**: el método que envía tu prompt y recibe la respuesta
- **`model`**: qué modelo quieres usar (Gemma-3-4b, Gemini-2.0-flash, etc.)
- **`contents`**: tu prompt (la instrucción o pregunta)
- **`response.text`**: el texto de la respuesta

In [ ]:
# Ver todos los modelos disponibles en tu cuenta
# Útil para saber qué modelos puedes usar
from google import genai
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")
client = genai.Client(api_key=os.getenv("Gemini_API_Key_001"))

# Itera sobre todos los modelos disponibles y muestra su nombre
print("Modelos disponibles:")
for model in client.models.list():
    print(f"  {model.name}")

In [32]:
from google import genai
import os
from dotenv import load_dotenv

# Cargamos credenciales
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# Creamos el cliente: gestiona la autenticación y la conexión
client = genai.Client(api_key=os.getenv("Gemini_API_Key_001"))

# Llamada al modelo:
#   - model: identificador del modelo a usar
#   - contents: el prompt que enviamos
response = client.models.generate_content(
    model="models/gemma-3-4b-it",
    contents="Explica qué es un LLM en 3 líneas"
)

# response.text contiene el texto generado por el modelo
print(response.text)

Un LLM (Large Language Model, o Modelo de Lenguaje Grande) es un tipo de inteligencia artificial entrenado con enormes cantidades de texto para comprender y generar lenguaje humano.  Utiliza patrones aprendidos para responder preguntas, escribir textos creativos y traducir idiomas.  En esencia, son máquinas que "hablan" y "escriben" de manera sorprendentemente fluida.


## 5. ¿Qué es un prompt?

El **prompt** es el texto que envías al modelo. Es la instrucción, pregunta o contexto que le das para que genere una respuesta.

### Regla fundamental:
> **Más específico el prompt → mejor y más predecible la respuesta**

### Comparación:

| Prompt vago | Prompt específico |
|-------------|------------------|
| `"Explícame IA"` | `"Explica qué es la IA generativa en 3 líneas para alguien que sabe Machine Learning"` |
| `"Escribe un email"` | `"Escribe un email formal de disculpa a un cliente por un retraso en el pedido. Máximo 100 palabras."` |

### Componentes de un buen prompt:
1. **Rol** (opcional): *"Actúa como un experto en..."*
2. **Tarea**: qué quieres que haga
3. **Contexto**: información relevante
4. **Formato**: cómo quieres la respuesta (longitud, estructura, tono)

In [33]:
# Comparamos un prompt vago vs uno específico
# Observa cómo cambia la respuesta

print("=== PROMPT VAGO ===")
response_vago = client.models.generate_content(
    model="models/gemma-3-4b-it",
    contents='Explícame IA'   # Demasiado genérico
)
print(response_vago.text)
print("\n" + "="*50 + "\n")

print("=== PROMPT ESPECÍFICO ===")
response_especifico = client.models.generate_content(
    model="models/gemma-3-4b-it",
    contents='Explica qué es un LLM en 3 líneas para un principiante'  # Específico y acotado
)
print(response_especifico.text)

=== PROMPT VAGO ===


## 6. Limitaciones de los LLMs

Es fundamental conocer qué **NO pueden hacer** los LLMs para usarlos correctamente.

### 1. Alucinaciones
El modelo puede **inventar información** con total confianza. No verifica hechos — solo predice tokens probables.
- Ejemplo: dar una fecha incorrecta, inventar una cita, crear una referencia bibliográfica falsa
- **Solución**: siempre valida datos críticos con fuentes externas

### 2. Sin acceso a tus datos
El modelo solo conoce lo que se le dijo durante el entrenamiento (hasta una fecha de corte) y lo que le envías en el prompt.
- No sabe nada sobre tu empresa, tus documentos o datos privados
- **Solución**: incluir el contexto relevante en el prompt (técnica RAG)

### 3. Sin memoria persistente
Cada llamada a la API es independiente. El modelo no recuerda conversaciones anteriores.
- **Solución**: enviar el historial de mensajes en cada llamada

### 4. Sesgos
El modelo aprende de texto de internet, que contiene prejuicios humanos. Puede reproducirlos.

### 5. Fecha de corte
El modelo no conoce eventos posteriores a su fecha de entrenamiento.

## 7. Flujo básico de uso

### Uso simple (este notebook):
```
Tu prompt → LLM → Respuesta en texto
```

### Uso en aplicaciones reales:
```
Datos del sistema/usuario → Prompt construido dinámicamente → LLM → Resultado procesado → Acción
```

### Ejemplo en una empresa:
```
Reseña del cliente (dato) 
    → Prompt: "Clasifica esta reseña como positiva/negativa: {reseña}" 
    → LLM 
    → "NEGATIVA" 
    → Dispara alerta al equipo de soporte
```

### Resumen de lo aprendido:

| Concepto | Descripción |
|----------|-------------|
| **LLM** | Modelo que predice texto token a token |
| **API Key** | Credencial para autenticarte con el servicio |
| **.env** | Archivo donde guardas claves de forma segura |
| **`genai.Client`** | Objeto que gestiona la conexión con la API |
| **`generate_content()`** | Método para enviar un prompt y recibir respuesta |
| **Prompt** | Instrucción que envías al modelo |
| **`response.text`** | Texto generado por el modelo |